# Rate scenarios

Parallel shifts and key-rate twists on discount (and optional forward) curves: steepener, flattener, and inversion-style shapes. Built-ins are used when present; custom `build_scenario_spec` operations fill the gaps.

## Concept

- **Parallel shift** moves all zero (or par-equivalent) rates by the same amount; discount factors respond uniformly across maturities.
- **Steepener** lifts long-end rates relative to the short end (or the reverse for a **flattener**).
- **Inversion** is a stylized shape where short policy rates sit above long forwards—here we approximate it with **node shocks** that lift the front end and ease the belly/long end.

Shocks are authored with the typed builders `OperationSpec.curve_parallel_bp` and `OperationSpec.curve_node_bp`, choosing the curve family with `CurveKind.discount()` (or `CurveKind.forward()`, shown at the end) and the tenor alignment with `TenorMatchMode.interpolate()`. Validation runs through `validate_scenario_spec`.

In [1]:
import json

from finstack_quant.scenarios import (
    CurveKind,
    OperationSpec,
    TenorMatchMode,
    apply_scenario_to_market,
    build_from_template,
    build_scenario_spec,
    build_template_component,
    compose_scenarios,
    list_builtin_templates,
    list_template_components,
    validate_scenario_spec,
)

templates = list_builtin_templates()
print("Built-in templates (rate-related may include rate_shock_2022):", templates)

rate_tpl_json = None
if "rate_shock_2022" in templates:
    rate_tpl_json = build_from_template("rate_shock_2022")
    print("Loaded template rate_shock_2022; spec length:", len(rate_tpl_json))
    comps = list_template_components("rate_shock_2022")
    print("Components:", comps)
    if comps:
        c0 = build_template_component("rate_shock_2022", comps[0])
        print("First component spec length:", len(c0))
else:
    print("Template rate_shock_2022 not in this build; relying on custom ops only.")


def ops_json(*ops: OperationSpec) -> str:
    """Serialize typed operations into the JSON list `build_scenario_spec` expects."""
    return json.dumps([json.loads(op.to_json()) for op in ops])


DISCOUNT = CurveKind.discount()
INTERPOLATE = TenorMatchMode.interpolate()
OIS = "USD-OIS"

# The four rate shapes, authored once with the typed `OperationSpec` builders.
# The application cell below reuses this dict rather than re-declaring the shocks.
SPECS = {
    "+100bp parallel": build_scenario_spec(
        "parallel_plus_100bp",
        ops_json(OperationSpec.curve_parallel_bp(DISCOUNT, OIS, 100.0)),
        "+100bp parallel",
        "Uniform +100bp on USD-OIS discount curve",
    ),
    # Short end ~flat, long end higher (bear steepener in rate space).
    "Steepener": build_scenario_spec(
        "steepener",
        ops_json(
            OperationSpec.curve_node_bp(DISCOUNT, OIS, [("6M", 0.0), ("5Y", 60.0)], INTERPOLATE)
        ),
        "Steepener",
        "Short flat, long rates up (key-rate style)",
    ),
    # Short end up, long end ~unchanged.
    "Flattener": build_scenario_spec(
        "flattener",
        ops_json(
            OperationSpec.curve_node_bp(DISCOUNT, OIS, [("6M", 55.0), ("5Y", 5.0)], INTERPOLATE)
        ),
        "Flattener",
        "Short up, long anchored",
    ),
    # Stylized inversion: front end up strongly, long end down.
    "Inversion-style": build_scenario_spec(
        "inversion_style",
        ops_json(
            OperationSpec.curve_node_bp(
                DISCOUNT, OIS, [("3M", 90.0), ("2Y", 40.0), ("5Y", -25.0)], INTERPOLATE
            )
        ),
        "Inversion-style twist",
        "Front end up, long end down (illustrative)",
    ),
}

for label, spec_json in SPECS.items():
    ok = validate_scenario_spec(spec_json)
    print(f"Scenario {label!r} valid={ok}; json chars={len(spec_json)}")

merged = compose_scenarios(
    json.dumps([json.loads(SPECS["+100bp parallel"]), json.loads(SPECS["Steepener"])])
)
print(
    "Composed parallel+steepener:",
    validate_scenario_spec(merged),
    "id=",
    json.loads(merged).get("id"),
    "ops=",
    len(json.loads(merged).get("operations", [])),
)
print("apply_scenario_to_market callable:", callable(apply_scenario_to_market))

if rate_tpl_json:
    print("Historical template snippet:", rate_tpl_json[:240], "...")


Built-in templates (rate-related may include rate_shock_2022): ['gfc_2008', 'covid_2020', 'rate_shock_2022', 'svb_2023', 'ltcm_1998']
Loaded template rate_shock_2022; spec length: 727
Components: ['rate_shock_2022_rates', 'rate_shock_2022_credit', 'rate_shock_2022_equity', 'rate_shock_2022_vol', 'rate_shock_2022_fx']
First component spec length: 235
Scenario '+100bp parallel' valid=True; json chars=262
Scenario 'Steepener' valid=True; json chars=293
Scenario 'Flattener' valid=True; json chars=274
Scenario 'Inversion-style' valid=True; json chars=325
Composed parallel+steepener: True id= parallel_plus_100bp+steepener ops= 2
apply_scenario_to_market callable: True
Historical template snippet: {"id":"rate_shock_2022","name":"2022 Fed Hiking Cycle","description":"Inflation shock - rates surge, credit widens, equities fall, vol rises","operations":[{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-SOFR","bp":300.0 ...


**`match_mode: interpolate` on `curve_node_bp`.** Shocks are defined on **named tenor pillars** (for example **6M** and **5Y**). With **`interpolate`**, the **basis-point bump** between two pillars varies **linearly** in tenor between those pillars; **at or beyond** the endpoints, the bump is usually held **flat** at the nearest pillar’s level (so the short end gets the 6M bump and the long end the 5Y bump in a two-node example). This is **not** the same as interpolating **discount factors** themselves—the scenario layer adjusts **quotes / zero rates** per engine rules, then rebuilds the curve. If a pillar lies **outside** the curve’s domain, expect **extrapolation warnings** and **flat** extension off the last live pillar.

In [2]:
import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF
from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext

bd = DEMO_AS_OF
AS_OF = str(bd)

# A deliberately short (5Y) OIS curve plus a 3M forward curve. The compact pillar
# set is what makes the key-rate interpolation between the 6M and 5Y nodes easy to
# read below; `_shared.build_demo_market` carries a longer, richer term structure.
ois_knots = [(0.0, 1.0), (0.5, 0.994), (1.0, 0.985), (2.0, 0.965), (5.0, 0.88)]
mc = MarketContext()
mc.insert(DiscountCurve("USD-OIS", bd, ois_knots, day_count="act_365f"))
mc.insert(
    ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.045), (1.0, 0.048), (5.0, 0.052)],
        bd,
        day_count="act_360",
    )
)
base_json = mc.to_json()
print("Base market built; JSON length:", len(base_json))

def print_df_pair(label: str, base_curve: DiscountCurve, stressed_mj: str):
    stressed_context = MarketContext.from_json(stressed_mj)
    stressed_curve = stressed_context.get_discount(base_curve.id)
    for t in (0.5, 1.0, 2.0, 5.0):
        b = base_curve.df(t)
        a = stressed_curve.df(t)
        print(f"  {label} t={t}Y  DF {b:.6f} -> {a:.6f}  (delta {a - b:+.6f})")

base_dc = mc.get_discount("USD-OIS")
print("Base implied zero rates (curve.zero(t), same pillars as DF table):")
for t in (0.25, 0.5, 1.0, 2.0, 5.0):
    print(f"  t={t}Y  zero={base_dc.zero(t):.6f}")

# Reuse the four specs authored above — no need to re-declare the shocks here.
for name, spec in SPECS.items():
    out = apply_scenario_to_market(spec, base_json, AS_OF)
    print(f"\n=== {name} ===")
    print("  operations_applied:", out["operations_applied"])
    print("  warnings:", out["warnings"])
    print_df_pair("USD-OIS", base_dc, out["market_json"])
    if name == "Steepener":
        stressed_context = MarketContext.from_json(out["market_json"])
        stressed_curve = stressed_context.get_discount("USD-OIS")
        print("  Implied zero rates: base -> steepener (illustrates interpolate between 6M and 5Y nodes)")
        for t in (0.25, 0.5, 1.0, 2.0, 5.0):
            print(f"    t={t}Y  {base_dc.zero(t):.6f} -> {stressed_curve.zero(t):.6f}")

# Forward curve: small parallel bump to show ForwardCurve in the pipeline.
# `CurveKind.forward()` selects the forward-curve family instead of discount.
fwd_spec = build_scenario_spec(
    "fwd_bump",
    ops_json(OperationSpec.curve_parallel_bp(CurveKind.forward(), "USD-SOFR-3M", 15.0)),
    "Forward +15bp",
    "Parallel +15bp on the USD-SOFR-3M forward curve",
)
fr = apply_scenario_to_market(fwd_spec, base_json, AS_OF)
print("\n=== Forward +15bp parallel (USD-SOFR-3M) ===")
print("  operations_applied:", fr["operations_applied"])
stressed_forward = MarketContext.from_json(fr["market_json"]).get_forward("USD-SOFR-3M")
print("  forward rate at t=0 after shock:", stressed_forward.rate(0.0))


Base market built; JSON length: 1518
Base implied zero rates (curve.zero(t), same pillars as DF table):
  t=0.25Y  zero=0.010497
  t=0.5Y  zero=0.012036
  t=1.0Y  zero=0.015114
  t=2.0Y  zero=0.017814
  t=5.0Y  zero=0.025567

=== +100bp parallel ===
  operations_applied: 1
  warnings: []
  USD-OIS t=0.5Y  DF 0.994000 -> 0.989042  (delta -0.004958)
  USD-OIS t=1.0Y  DF 0.985000 -> 0.975199  (delta -0.009801)
  USD-OIS t=2.0Y  DF 0.965000 -> 0.945892  (delta -0.019108)
  USD-OIS t=5.0Y  DF 0.880000 -> 0.837082  (delta -0.042918)

=== Steepener ===
  operations_applied: 1
  warnings: []
  USD-OIS t=0.5Y  DF 0.994000 -> 0.994000  (delta +0.000000)
  USD-OIS t=1.0Y  DF 0.985000 -> 0.985000  (delta +0.000000)
  USD-OIS t=2.0Y  DF 0.965000 -> 0.965000  (delta +0.000000)
  USD-OIS t=5.0Y  DF 0.880000 -> 0.853992  (delta -0.026008)
  Implied zero rates: base -> steepener (illustrates interpolate between 6M and 5Y nodes)
    t=0.25Y  0.010497 -> 0.010497
    t=0.5Y  0.012036 -> 0.012036
    t=1.

## Takeaways

- **`OperationSpec.curve_parallel_bp`** is the workhorse for parallel DV01-style shocks; pick the family with **`CurveKind.discount()`** or **`CurveKind.forward()`**.
- **`OperationSpec.curve_node_bp`** with **`TenorMatchMode.interpolate()`** implements **key-rate** style shapes: steepener, flattener, and inversion-like twists.
- Author each shock **once** as a `ScenarioSpec` and reuse the spec string across cells — specs are just JSON, so they are cheap to keep in a dict, log, or diff.
- **`apply_scenario_to_market`** returns stressed **`market_json`**; deserialize it with **`MarketContext.from_json`**, then use **`get_discount`** or **`get_forward`** for typed curve inspection.
- Built-in **historical templates** may include a dedicated rates pack—check membership in `list_builtin_templates()` since template IDs differ by build.